# 06_rnn_lstm_gru: Bidirectional Recurrent Sequence Classifiers
    
This notebook trains a recurrent classifier in PyTorch to classify sentence lengths (long vs. short) using vocabulary loaded from Gutenberg's *Alice in Wonderland*.


In [1]:
import re
import nltk
import torch

# Load Alice in Wonderland sentences
nltk.download('gutenberg', quiet=True)
from nltk.corpus import gutenberg
sentences_raw = gutenberg.sents('carroll-alice.txt')

cleaned_sentences = []
for s in sentences_raw:
    words = [w.lower() for w in s if re.match(r"^\w+$", w)]
    if 3 < len(words) < 25:
        cleaned_sentences.append(words)

vocab = {"<pad>": 0, "<unk>": 1}
for s in cleaned_sentences[:500]:
    for w in s:
        if w not in vocab:
            vocab[w] = len(vocab)
vocab_size = len(vocab)
print("Vocabulary Size:", vocab_size)


Vocabulary Size: 1074


### Output Explanation: Vocab Mapping
- **Tokens Mapping**: Maps words to vocabulary index dictionaries, setting `<pad>` to index 0 and `<unk>` to index 1.


In [2]:
seq_len = 20
X_data = []
y_data = []

for s in cleaned_sentences[:300]:
    indices = [vocab.get(w, 1) for w in s]
    if len(indices) < seq_len:
        indices = indices + [0] * (seq_len - len(indices))
    else:
        indices = indices[:seq_len]
    X_data.append(indices)
    # Binary classification target: sentence length > 12 tokens
    y_data.append(1 if len(s) > 12 else 0)

X = torch.tensor(X_data, dtype=torch.long)
y = torch.tensor(y_data, dtype=torch.long)

print("Input X tensor shape:", X.shape)
print("Target y tensor shape:", y.shape)


Input X tensor shape: torch.Size([300, 20])
Target y tensor shape: torch.Size([300])


### Output Explanation: Padded Input Tensors
- **Dimensions**: The inputs `X` have the shape `(300, 20)`, representing 300 batch sequences padded or sliced to a length of 20.
- **Targets**: `y` is a binary label tensor of shape `(300,)`.


In [3]:
import torch.nn as nn
import torch.optim as optim

embedding_dim = 16
hidden_dim = 24
num_classes = 2

class RecurrentClassifier(nn.Module):
    def __init__(self, cell_type="LSTM"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if cell_type == "RNN":
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "LSTM":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "GRU":
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded)
        # Slices final hidden state of bidir layers
        last_step = out[:, -1, :]
        return self.fc(last_step)

print("Classifier architectures defined successfully.")


Classifier architectures defined successfully.


### Output Explanation: Model Definitions
- **Bidirectional Layer**: We set `bidirectional=True` in PyTorch, which runs two independent hidden layers (forward and backward). The final linear classification layer receives the concatenated representations of shape `(batch, hidden_dim * 2)`.


In [4]:
for cell_name in ["RNN", "LSTM", "GRU"]:
    model = RecurrentClassifier(cell_type=cell_name)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    # Run 5 training epochs
    for epoch in range(5):
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"{cell_name} Classifier Final Loss: {loss.item():.4f}")


RNN Classifier Final Loss: 0.5873
LSTM Classifier Final Loss: 0.6372
GRU Classifier Final Loss: 0.5746


### Output Explanation: Training Comparison
- **Final Loss**: Shows training losses across 5 epochs. In general, LSTMs and GRUs show more stable loss decay on long context sequences compared to standard RNN cells.
